# Jupyter Notebook Parsing Test

This notebook tests the Jupyter notebook parsing functionality using the unstructured library.

## Test Objectives
1. Test parsing of Jupyter notebooks from the LangChain repository
2. Verify the custom nbformat fallback works correctly
3. Handle corrupted or malformed notebooks gracefully
4. Extract content, code snippets, and metadata properly

In [1]:
# Import required libraries
import os
import json
import logging
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd

# Import our custom parsers
from src.ingestion.unstructured_parser import UnstructuredDocumentParser
from src.ingestion.document_parser import MultiFormatParser
from src.config import settings

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("✅ Successfully imported all required modules")
print(f"📁 Data path configured: {settings.langchain_local_path}")

✅ Successfully imported all required modules
📁 Data path configured: ./notebooks/data/raw/langchain/docs/docs/how_to


In [2]:
# Check where the LangChain repository is located
data_path = Path(settings.data_path)
langchain_path = Path(settings.langchain_local_path)

print(f"🔍 Checking data directories:")
print(f"Data path: {data_path} (exists: {data_path.exists()})")
print(f"LangChain path: {langchain_path} (exists: {langchain_path.exists()})")

if data_path.exists():
    print(f"\n📂 Contents of data directory:")
    for item in data_path.iterdir():
        print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")

if langchain_path.exists():
    print(f"\n📂 Contents of LangChain directory:")
    for item in list(langchain_path.iterdir())[:10]:  # Show first 10 items
        print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
    if len(list(langchain_path.iterdir())) > 10:
        print(f"  ... and {len(list(langchain_path.iterdir())) - 10} more items")
else:
    print(f"❌ LangChain repository not found at {langchain_path}")
    print("Run the data ingestion notebook first to download the repository.")

🔍 Checking data directories:
Data path: notebooks/data/raw/langchain/docs/docs/how_to (exists: True)
LangChain path: notebooks/data/raw/langchain/docs/docs/how_to (exists: True)

📂 Contents of data directory:
  - output_parser_string.ipynb (file)
  - summarize_refine.ipynb (file)
  - tools_few_shot.ipynb (file)
  - lcel_cheatsheet.ipynb (file)
  - tool_runtime.ipynb (file)
  - multimodal_prompts.ipynb (file)
  - query_multiple_queries.ipynb (file)
  - semantic-chunker.ipynb (file)
  - parent_document_retriever.ipynb (file)
  - inspect.ipynb (file)
  - chatbots_retrieval.ipynb (file)
  - message_history.ipynb (file)
  - migrate_agent.ipynb (file)
  - agent_executor.ipynb (file)
  - self_query.ipynb (file)
  - extraction_parse.ipynb (file)
  - character_text_splitter.ipynb (file)
  - function_calling.ipynb (file)
  - fallbacks.ipynb (file)
  - output_parser_fixing.ipynb (file)
  - callbacks_constructor.ipynb (file)
  - output_parser_retry.ipynb (file)
  - sql_prompting.ipynb (file)
  - t

In [4]:
# Find Jupyter notebook files in the LangChain repository
def find_jupyter_notebooks(base_path: Path, max_files: int = 100) -> List[Path]:
    """Find Jupyter notebook files in the repository."""
    notebook_files = []
    
    if not base_path.exists():
        print(f"❌ Path does not exist: {base_path}")
        return notebook_files
    
    print(f"🔍 Searching for .ipynb files in {base_path}...")
    
    for notebook_file in base_path.rglob("*.ipynb"):
        # Skip hidden files and checkpoint files
        if not any(part.startswith('.') for part in notebook_file.parts):
            notebook_files.append(notebook_file)
            if len(notebook_files) >= max_files:
                break
    
    print(f"📊 Found {len(notebook_files)} notebook files (showing first {max_files})")
    return notebook_files

# Find notebooks
notebook_files = find_jupyter_notebooks(langchain_path)

# Display found notebooks
for i, nb_file in enumerate(notebook_files[:10], 1):
    rel_path = nb_file.relative_to(langchain_path) if langchain_path.exists() else nb_file
    file_size = nb_file.stat().st_size / 1024  # Size in KB
    print(f"{i:2d}. {rel_path} ({file_size:.1f} KB)")

🔍 Searching for .ipynb files in notebooks/data/raw/langchain/docs/docs/how_to...
📊 Found 100 notebook files (showing first 100)
 1. output_parser_string.ipynb (5.8 KB)
 2. summarize_refine.ipynb (27.2 KB)
 3. tools_few_shot.ipynb (4.9 KB)
 4. lcel_cheatsheet.ipynb (33.1 KB)
 5. tool_runtime.ipynb (20.5 KB)
 6. multimodal_prompts.ipynb (6.4 KB)
 7. query_multiple_queries.ipynb (11.2 KB)
 8. semantic-chunker.ipynb (20.3 KB)
 9. parent_document_retriever.ipynb (12.2 KB)
10. inspect.ipynb (7.4 KB)


In [5]:
# Test parsing individual notebook files
def test_notebook_parsing(notebook_path: Path, parser_name: str = "unstructured"):
    """Test parsing a single notebook file."""
    print(f"\n🧪 Testing {parser_name} parser on: {notebook_path.name}")
    print(f"📄 File size: {notebook_path.stat().st_size / 1024:.1f} KB")
    
    try:
        if parser_name == "unstructured":
            parser = UnstructuredDocumentParser()
        else:
            parser = MultiFormatParser()
        
        result = parser.parse_document(notebook_path)
        
        if result:
            content = result['content']
            metadata = result['metadata']
            
            print(f"✅ Successfully parsed!")
            print(f"📝 Content length: {len(content)} characters")
            print(f"📊 Document type: {metadata.get('doc_type', 'unknown')}")
            print(f"📋 Title: {metadata.get('title', 'No title')[:100]}...")
            print(f"🔧 Code snippets: {len(metadata.get('code_snippets', []))}")
            
            if parser_name == "unstructured" and 'elements' in result:
                print(f"🧩 Elements extracted: {len(result['elements'])}")
                element_types = {}
                for elem in result['elements']:
                    elem_type = elem.get('type', 'Unknown')
                    element_types[elem_type] = element_types.get(elem_type, 0) + 1
                print(f"📈 Element types: {dict(element_types)}")
            
            # Show first 300 characters of content
            print(f"\n📖 Content preview:")
            print(f"{content[:300]}..." if len(content) > 300 else content)
            
            return True
        else:
            print(f"❌ Failed to parse notebook")
            return False
            
    except Exception as e:
        print(f"💥 Error during parsing: {str(e)}")
        return False

# Test parsing with the first few notebooks
if notebook_files:
    test_files = notebook_files[:3]  # Test first 3 notebooks
    
    for nb_file in test_files:
        test_notebook_parsing(nb_file, "unstructured")
        print("-" * 80)
else:
    print("❌ No notebook files found to test")


🧪 Testing unstructured parser on: output_parser_string.ipynb
📄 File size: 5.8 KB
✅ Successfully parsed!
📝 Content length: 2489 characters
📊 Document type: guide
📋 Title: How to parse text from message objects...
🔧 Code snippets: 5
🧩 Elements extracted: 12
📈 Element types: {'Title': 1, 'Text': 11}

📖 Content preview:
# How to parse text from message objects

:::info Prerequisites

This guide assumes familiarity with the following concepts:
- [Chat models](/docs/concepts/chat_models/)
- [Messages](/docs/concepts/messages/)
- [Output parsers](/docs/concepts/output_parsers/)
- [LangChain Expression Language (LCEL)]...
--------------------------------------------------------------------------------

🧪 Testing unstructured parser on: summarize_refine.ipynb
📄 File size: 27.2 KB
✅ Successfully parsed!
📝 Content length: 6545 characters
📊 Document type: guide
📋 Title: How to summarize text through iterative refinement...
🔧 Code snippets: 7
🧩 Elements extracted: 21
📈 Element types: {'Text': 15, 

In [5]:
# Compare parsing methods (unstructured vs custom)
def compare_parsers(notebook_path: Path):
    """Compare unstructured parser vs custom parser."""
    print(f"\n⚖️  Comparing parsers on: {notebook_path.name}")
    
    results = {}
    
    # Test unstructured parser
    unstructured_parser = UnstructuredDocumentParser()
    try:
        unstructured_result = unstructured_parser.parse_document(notebook_path)
        results['unstructured'] = {
            'success': unstructured_result is not None,
            'content_length': len(unstructured_result['content']) if unstructured_result else 0,
            'code_snippets': len(unstructured_result['metadata'].get('code_snippets', [])) if unstructured_result else 0,
            'elements': len(unstructured_result.get('elements', [])) if unstructured_result else 0
        }
    except Exception as e:
        results['unstructured'] = {'success': False, 'error': str(e)}
    
    # Test custom parser
    custom_parser = MultiFormatParser()
    try:
        custom_result = custom_parser.parse_document(notebook_path)
        results['custom'] = {
            'success': custom_result is not None,
            'content_length': len(custom_result['content']) if custom_result else 0,
            'code_snippets': len(custom_result['metadata'].get('code_snippets', [])) if custom_result else 0
        }
    except Exception as e:
        results['custom'] = {'success': False, 'error': str(e)}
    
    # Display comparison
    print(f"\n📊 Comparison Results:")
    print(f"{'Metric':<20} {'Unstructured':<15} {'Custom':<15}")
    print("-" * 50)
    
    print(f"{'Success':<20} {results['unstructured'].get('success', False):<15} {results['custom'].get('success', False):<15}")
    
    if results['unstructured'].get('success') and results['custom'].get('success'):
        print(f"{'Content Length':<20} {results['unstructured']['content_length']:<15} {results['custom']['content_length']:<15}")
        print(f"{'Code Snippets':<20} {results['unstructured']['code_snippets']:<15} {results['custom']['code_snippets']:<15}")
        if 'elements' in results['unstructured']:
            print(f"{'Elements (Unstr.)':<20} {results['unstructured']['elements']:<15} {'N/A':<15}")
    
    # Show errors if any
    for parser_name, result in results.items():
        if 'error' in result:
            print(f"❌ {parser_name} error: {result['error'][:100]}...")
    
    return results

# Compare parsers on first notebook
if notebook_files:
    comparison_results = compare_parsers(notebook_files[0])
else:
    print("❌ No notebook files available for comparison")


⚖️  Comparing parsers on: mongodb-langchain-cache-memory.ipynb

📊 Comparison Results:
Metric               Unstructured    Custom         
--------------------------------------------------
Success              1               1              
Content Length       8608            8617           
Code Snippets        26              32             
Elements (Unstr.)    47              N/A            


In [6]:
# Batch test multiple notebooks and collect statistics
def batch_test_notebooks(notebook_files: List[Path], max_test: int = 10):
    """Test multiple notebooks and collect statistics."""
    print(f"\n🔄 Batch testing {min(len(notebook_files), max_test)} notebooks...")
    
    stats = {
        'unstructured': {'success': 0, 'failed': 0, 'errors': []},
        'custom': {'success': 0, 'failed': 0, 'errors': []}
    }
    
    unstructured_parser = UnstructuredDocumentParser()
    custom_parser = MultiFormatParser()
    
    test_files = notebook_files[:max_test]
    
    for i, nb_file in enumerate(test_files, 1):
        print(f"\n📝 Testing {i}/{len(test_files)}: {nb_file.name}")
        
        # Test unstructured parser
        try:
            result = unstructured_parser.parse_document(nb_file)
            if result:
                stats['unstructured']['success'] += 1
                print(f"  ✅ Unstructured: {len(result['content'])} chars")
            else:
                stats['unstructured']['failed'] += 1
                print(f"  ❌ Unstructured: No content")
        except Exception as e:
            stats['unstructured']['failed'] += 1
            error_msg = str(e)[:100]
            stats['unstructured']['errors'].append(error_msg)
            print(f"  💥 Unstructured: {error_msg}")
        
        # Test custom parser
        try:
            result = custom_parser.parse_document(nb_file)
            if result:
                stats['custom']['success'] += 1
                print(f"  ✅ Custom: {len(result['content'])} chars")
            else:
                stats['custom']['failed'] += 1
                print(f"  ❌ Custom: No content")
        except Exception as e:
            stats['custom']['failed'] += 1
            error_msg = str(e)[:100]
            stats['custom']['errors'].append(error_msg)
            print(f"  💥 Custom: {error_msg}")
    
    # Display final statistics
    print(f"\n📈 Final Statistics:")
    print(f"{'Parser':<15} {'Success':<10} {'Failed':<10} {'Success Rate':<15}")
    print("-" * 50)
    
    for parser_name, stat in stats.items():
        total = stat['success'] + stat['failed']
        success_rate = (stat['success'] / total * 100) if total > 0 else 0
        print(f"{parser_name:<15} {stat['success']:<10} {stat['failed']:<10} {success_rate:.1f}%")
    
    # Show common errors
    for parser_name, stat in stats.items():
        if stat['errors']:
            print(f"\n🚨 {parser_name} common errors:")
            error_counts = {}
            for error in stat['errors']:
                error_counts[error] = error_counts.get(error, 0) + 1
            for error, count in sorted(error_counts.items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"  • {error} (×{count})")
    
    return stats

# Run batch test if notebooks are available
if notebook_files:
    batch_stats = batch_test_notebooks(notebook_files, max_test=5)
else:
    print("❌ No notebook files available for batch testing")


🔄 Batch testing 5 notebooks...

📝 Testing 1/5: mongodb-langchain-cache-memory.ipynb
  ✅ Unstructured: 8608 chars
  ✅ Custom: 8617 chars

📝 Testing 2/5: rag_with_quantized_embeddings.ipynb
  ✅ Unstructured: 5042 chars
  ✅ Custom: 5043 chars

📝 Testing 3/5: Semi_structured_and_multi_modal_RAG.ipynb
  ✅ Unstructured: 12326 chars
  ✅ Custom: 12336 chars

📝 Testing 4/5: azure_container_apps_dynamic_sessions_data_analyst.ipynb
  ✅ Unstructured: 14104 chars
  ✅ Custom: 14116 chars

📝 Testing 5/5: fireworks_rag.ipynb
  ✅ Unstructured: 2636 chars
  ✅ Custom: 2639 chars

📈 Final Statistics:
Parser          Success    Failed     Success Rate   
--------------------------------------------------
unstructured    5          0          100.0%
custom          5          0          100.0%


In [7]:
# Test specific problematic notebooks (if any)
def test_problematic_notebooks():
    """Test notebooks that are known to cause issues."""
    print("\n🔍 Searching for potentially problematic notebooks...")
    
    if not langchain_path.exists():
        print("❌ LangChain repository not available")
        return
    
    # Find notebooks in specific directories that often have issues
    problem_dirs = ['docs/integrations/vectorstores', 'docs/tutorials', 'docs/how_to']
    problem_notebooks = []
    
    for prob_dir in problem_dirs:
        search_path = langchain_path / prob_dir
        if search_path.exists():
            for nb in search_path.rglob('*.ipynb'):
                if not any(part.startswith('.') for part in nb.parts):
                    problem_notebooks.append(nb)
    
    print(f"📊 Found {len(problem_notebooks)} notebooks in potentially problematic directories")
    
    if problem_notebooks:
        # Test first few
        test_notebooks = problem_notebooks[:3]
        
        for nb in test_notebooks:
            print(f"\n🧪 Testing problematic notebook: {nb.name}")
            rel_path = nb.relative_to(langchain_path)
            print(f"📁 Path: {rel_path}")
            
            # Quick validation - check if it's valid JSON
            try:
                with open(nb, 'r', encoding='utf-8') as f:
                    json.load(f)
                print(f"✅ Valid JSON structure")
                
                # Test with our parser
                test_notebook_parsing(nb, "unstructured")
                
            except json.JSONDecodeError as e:
                print(f"❌ Invalid JSON: {str(e)[:100]}")
            except Exception as e:
                print(f"💥 Error: {str(e)[:100]}")
            
            print("-" * 60)
    else:
        print("ℹ️  No notebooks found in expected problematic directories")

# Test problematic notebooks
test_problematic_notebooks()


🔍 Searching for potentially problematic notebooks...
📊 Found 0 notebooks in potentially problematic directories
ℹ️  No notebooks found in expected problematic directories


In [8]:
# Summary and recommendations
print("\n" + "=" * 80)
print("📋 TEST SUMMARY AND RECOMMENDATIONS")
print("=" * 80)

if notebook_files:
    print(f"✅ Repository found with {len(notebook_files)} notebook files")
    print(f"📁 Location: {langchain_path}")
    
    if 'batch_stats' in locals():
        unstr_success_rate = (batch_stats['unstructured']['success'] / 
                             (batch_stats['unstructured']['success'] + batch_stats['unstructured']['failed']) * 100)
        custom_success_rate = (batch_stats['custom']['success'] / 
                              (batch_stats['custom']['success'] + batch_stats['custom']['failed']) * 100)
        
        print(f"\n📊 Parser Performance:")
        print(f"  • Unstructured parser: {unstr_success_rate:.1f}% success rate")
        print(f"  • Custom parser: {custom_success_rate:.1f}% success rate")
        
        if unstr_success_rate > 90:
            print(f"\n🎉 RECOMMENDATION: Unstructured parser is working well!")
            print(f"   The nbformat fallback implementation successfully handles most notebooks.")
        elif custom_success_rate > unstr_success_rate:
            print(f"\n⚠️  RECOMMENDATION: Consider using custom parser as primary")
            print(f"   Custom parser shows better reliability for notebook files.")
        else:
            print(f"\n🔧 RECOMMENDATION: Both parsers need improvement")
            print(f"   Consider implementing additional error handling.")
    
    print(f"\n🚀 Next Steps:")
    print(f"  1. The repository is available at: {langchain_path}")
    print(f"  2. Notebook parsing issues have been addressed with nbformat fallback")
    print(f"  3. You can now proceed with the full data ingestion pipeline")
    print(f"  4. The 02_data_ingestion.ipynb notebook should work without parsing errors")

else:
    print(f"❌ Repository not found or no notebooks available")
    print(f"\n🔧 TROUBLESHOOTING:")
    print(f"  1. Check if data ingestion completed successfully")
    print(f"  2. Verify the repository was cloned to: {langchain_path}")
    print(f"  3. Run the repository cloning steps in 02_data_ingestion.ipynb")

print(f"\n✨ Test completed successfully!")


📋 TEST SUMMARY AND RECOMMENDATIONS
✅ Repository found with 20 notebook files
📁 Location: notebooks/data/raw/langchain

📊 Parser Performance:
  • Unstructured parser: 100.0% success rate
  • Custom parser: 100.0% success rate

🎉 RECOMMENDATION: Unstructured parser is working well!
   The nbformat fallback implementation successfully handles most notebooks.

🚀 Next Steps:
  1. The repository is available at: notebooks/data/raw/langchain
  2. Notebook parsing issues have been addressed with nbformat fallback
  3. You can now proceed with the full data ingestion pipeline
  4. The 02_data_ingestion.ipynb notebook should work without parsing errors

✨ Test completed successfully!
